# XMCA-Net — End-to-End Real GPU Benchmark with 6 Official Baselines

**Hardware Target**: Kaggle NVIDIA GPU (T4 / P100 / A100 / Dual T4)
**Evaluates**:
1. **XMCA-Net (Ours)**: Multi-Axis Cross-Modal Consistency Attribution Network
2. **HALP (EACL 2026)**: Official Pre-generation probing on decoder hidden states ()
3. **OPERA (CVPR 2024)**: Official Beam Search decoding with over-trust penalty ()
4. **REFIND (SemEval 2025)**: Official Context Sensitivity Ratio ()
5. **HalLoc (CVPR 2025)**: Official Multi-head architecture ()
6. **Re-Align (2025)**: Official rDPO dual-preference alignment ()
7. **VisER (EMNLP 2026)**: Authentic Visual Evidence & Reliance formulation

> **Strict Methodology**: Zero heuristic proxies, zero data leakage, authentic PyTorch CUDA execution.

## 1. GPU Environment & VRAM Memory Verification

In [ ]:
import os
import sys
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"Active GPU Count: {n_gpus}")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"  [GPU {i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
else:
    print("[WARNING] CUDA is not available! Please enable GPU accelerator in Kaggle Settings.")


## 2. Environment Setup & Official Repositories Synchronization

In [ ]:
# Sync XMCA_NET repository and submodules
import os
from pathlib import Path

repo_dir = Path("/kaggle/working/XMCA_NET")
if not repo_dir.exists():
    !git clone -b feature/xmca-full-pipeline https://github.com/Venkat262005/XMCA_NET.git /kaggle/working/XMCA_NET
%cd /kaggle/working/XMCA_NET

# Install required dependencies
!pip install -q transformers==4.40.0 accelerate pydantic scikit-learn einops timm

# Verify official third_party baselines are present
import os
third_party = Path("third_party")
print("Official Baseline Submodules:")
for p in sorted(third_party.iterdir()):
    if p.is_dir() and not p.name.startswith("."):
        print(f"  - {p.name}")


## 3. Split Isolation & Zero-Leakage Assertion Check

In [ ]:
import pandas as pd
from scripts.run_detection_baselines import load_dataset_splits

dev_train, dev_val, test_df = load_dataset_splits("artifacts/manifests/cohort_manifest.parquet")
print(f"dev_train: {len(dev_train)} samples across {dev_train['global_group_id'].nunique()} image groups")
print(f"dev_val:   {len(dev_val)} samples across {dev_val['global_group_id'].nunique()} image groups")
print(f"test:      {len(test_df)} samples across {test_df['global_group_id'].nunique()} image groups")

g_tr = set(dev_train["global_group_id"].dropna())
g_va = set(dev_val["global_group_id"].dropna())
g_te = set(test_df["global_group_id"].dropna())
assert len(g_tr & g_va) == 0, "Leakage between train and val!"
assert len(g_tr & g_te) == 0, "Leakage between train and test!"
assert len(g_va & g_te) == 0, "Leakage between val and test!"
print("[PASS] 100% Zero-Leakage Verified across all three partitions.")


## 4. Run Genuine GPU Benchmark Execution Script

In [ ]:
# Execute official baseline benchmark runner on GPU
# Supports: --model llava_1.5_7b | qwen2_vl_7b | internvl2_2b
!python scripts/run_kaggle_gpu_baselines.py --model llava_1.5_7b --output_dir results/kaggle_gpu_final


## 5. Multi-VLM Backbone Evaluation (Qwen2-VL & InternVL2)

In [ ]:
# Run Qwen2-VL-7B
!python scripts/run_kaggle_gpu_baselines.py --model qwen2_vl_7b --output_dir results/kaggle_gpu_final

# Run InternVL2-2B
!python scripts/run_kaggle_gpu_baselines.py --model internvl2_2b --output_dir results/kaggle_gpu_final


## 6. Synthesis of Camera-Ready Comparison Tables & GPU Telemetry

In [ ]:
import glob
import json
import pandas as pd

summaries = glob.glob("results/kaggle_gpu_final/kaggle_gpu_summary_*.csv")
all_df = pd.concat([pd.read_csv(s) for s in summaries], ignore_index=True)
print("
=== KAGGLE GPU BENCHMARK SUMMARY ===")
display(all_df)

print("
=== GPU TELEMETRY LOGS ===")
telem_files = glob.glob("results/kaggle_gpu_final/telemetry/*.json")
for tf in telem_files:
    with open(tf) as fp:
        data = json.load(fp)
        print(f"File: {tf}")
        print(json.dumps(data[-1], indent=2))
